# Mistral OCR + EN 10204 fields

Active notebook **01**. Single API call per document: OCR + structured field extraction via Mistral Document AI `document_annotation_format`.

Use this for **cert metadata and field JSON**. For reliable table Min/Max vs result rows, use `notebooks/02-content-understanding-tables.ipynb`.

**Outputs (under `extraction-output/`):**
- `mistral-ocr/<cert>.md` — OCR markdown for cert pages
- `mistral-fields/<cert>.json` — structured EN 10204 fields

**Pipeline:**
1. OCR page 0 → detect cert type and primary cert page span
2. Skip types 2.1 / 2.2 (no test data)
3. OCR cert pages + extract fields in the same call
4. Save both outputs

See repo `README.md` for the full flow.


## Setup

In [ ]:
import base64
import json
import os
import re
import sys
from pathlib import Path
from urllib.parse import urlparse

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display

def find_repo_root() -> Path:
    """Resolve repo root whether the kernel cwd is repo root or notebooks/."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "raw-docs").is_dir() and (p / "scripts").is_dir():
            return p
    return here

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

# Azure Foundry endpoint in .env may include a project path; OCR uses the host + fixed path.
_base = urlparse(os.getenv("AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT", ""))
OCR_ENDPOINT = f"{_base.scheme}://{_base.netloc}/providers/mistral/azure/ocr"

KEY        = os.getenv("AZURE_MISTRAL_DOCUMENT_AI_KEY")
DEPLOYMENT = os.getenv("AZURE_AI_DEPLOYMENT_NAME")
HEADERS    = {"Content-Type": "application/json", "Authorization": f"Bearer {KEY}"}

INPUT_DIR      = ROOT / "raw-docs"
OUTPUT_DIR     = ROOT / "extraction-output"
OCR_DIR        = OUTPUT_DIR / "mistral-ocr"       # OCR markdown from this notebook
ANNOTATION_DIR = OUTPUT_DIR / "mistral-fields"    # structured EN 10204 JSON

for folder in [OCR_DIR, ANNOTATION_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Repo root:", ROOT)
print("Endpoint  :", OCR_ENDPOINT)
print("Model     :", DEPLOYMENT)
print("OCR out   :", OCR_DIR)
print("Fields out:", ANNOTATION_DIR)


## EN 10204 extraction schema

This schema tells Mistral Document AI exactly what fields to extract. The model fills this from the cert pages it just OCR'd — no second API call needed.

Fields cover all mandatory EN 10204 3.1 data:
- Certificate identification
- Product and order details  
- Chemical analysis (one row per heat — certs often have multiple heats)
- Mechanical properties (tensile, impact, hardness)
- Heat treatment
- Conformance and signatory

If a field is not in the document, the model returns null. Do not change null to a default — null means the cert is missing that field, which may be a non-conformance.

In [2]:
EN10204_SCHEMA = {
    "type": "object",
    "properties": {
        "cert_number":    {"type": ["string", "null"], "description": "Certificate number as printed on the document"},
        "cert_type":      {"type": ["string", "null"], "description": "EN 10204 type — e.g. 3.1, 3.2, 2.2"},
        "issue_date":     {"type": ["string", "null"], "description": "Date the certificate was issued"},
        "issuer":         {"type": ["string", "null"], "description": "Company that issued the certificate (manufacturer or distributor)"},
        "customer":       {"type": ["string", "null"], "description": "Customer name as shown on the certificate"},
        "purchase_order": {"type": ["string", "null"], "description": "Customer purchase order number"},
        "product": {
            "type": "object",
            "description": "Physical product being certified",
            "properties": {
                "form":          {"type": ["string", "null"], "description": "Product form e.g. bar, pipe, forging, casting, ring"},
                "grade":         {"type": ["string", "null"], "description": "Material grade name e.g. Alloy 625, F51, 316L"},
                "uns":           {"type": ["string", "null"], "description": "UNS number e.g. N06625, S31803"},
                "specification": {"type": ["array", "null"], "items": {"type": "string"}, "description": "All material specifications listed e.g. ASTM B446, ASME SB-446"},
                "dimensions":    {"type": ["string", "null"], "description": "Dimensions as printed e.g. 101.6mm OD x 1000mm long"},
                "quantity":      {"type": ["number", "null"], "description": "Number of pieces"}
            }
        },
        "melt_practice":  {"type": ["string", "null"], "description": "Melting process e.g. VIM/ESR, AOD, EAF"},
        "chemical_analysis": {
            "type": ["array", "null"],
            "description": "One entry per heat number. Certs from forging/casting manufacturers often have multiple heats.",
            "items": {
                "type": "object",
                "properties": {
                    "heat":  {"type": ["string", "null"], "description": "Heat or cast number"},
                    "C":     {"type": ["number", "null"]},
                    "Si":    {"type": ["number", "null"]},
                    "Mn":    {"type": ["number", "null"]},
                    "P":     {"type": ["number", "null"]},
                    "S":     {"type": ["number", "null"]},
                    "Cr":    {"type": ["number", "null"]},
                    "Ni":    {"type": ["number", "null"]},
                    "Mo":    {"type": ["number", "null"]},
                    "N":     {"type": ["number", "null"]},
                    "Nb_Ta": {"type": ["number", "null"], "description": "Nb+Ta combined"},
                    "Co":    {"type": ["number", "null"]},
                    "Cu":    {"type": ["number", "null"]},
                    "Fe":    {"type": ["number", "null"]},
                    "Al":    {"type": ["number", "null"]},
                    "Ti":    {"type": ["number", "null"]}
                }
            }
        },
        "mechanical_properties": {
            "type": ["array", "null"],
            "description": "Tensile test results. One entry per heat. Values are actual test results, not specification limits.",
            "items": {
                "type": "object",
                "properties": {
                    "heat":                {"type": ["string", "null"]},
                    "yield_mpa":           {"type": ["number", "null"], "description": "0.2% proof strength in MPa"},
                    "uts_mpa":             {"type": ["number", "null"], "description": "Ultimate tensile strength in MPa"},
                    "elongation_pct":      {"type": ["number", "null"], "description": "Elongation %"},
                    "reduction_area_pct":  {"type": ["number", "null"], "description": "Reduction of area %"}
                }
            }
        },
        "impact_test": {
            "type": ["array", "null"],
            "description": "Charpy impact test results. One entry per heat.",
            "items": {
                "type": "object",
                "properties": {
                    "heat":      {"type": ["string", "null"]},
                    "temp_c":    {"type": ["number", "null"], "description": "Test temperature in Celsius"},
                    "energy_j":  {"type": ["array", "null"],  "items": {"type": "number"}, "description": "Individual Joule readings"}
                }
            }
        },
        "hardness": {
            "type": ["array", "null"],
            "description": "Hardness test results. One entry per heat.",
            "items": {
                "type": "object",
                "properties": {
                    "heat":    {"type": ["string", "null"]},
                    "method":  {"type": ["string", "null"], "description": "e.g. HBW, HRC, HV"},
                    "values":  {"type": ["array", "null"], "items": {"type": "number"}}
                }
            }
        },
        "heat_treatment": {
            "type": ["object", "null"],
            "properties": {
                "type":    {"type": ["string", "null"], "description": "e.g. Anneal, Solution Anneal, Normalize, Quench and Temper"},
                "temp_c":  {"type": ["number", "null"], "description": "Treatment temperature in Celsius"},
                "soak":    {"type": ["string", "null"], "description": "Hold time e.g. 30 min, 1h"},
                "cooling": {"type": ["string", "null"], "description": "Cooling method e.g. Water Quench, Air Cool"}
            }
        },
        "grain_size":           {"type": ["string", "null"], "description": "ASTM grain size number if reported"},
        "conforming":           {"type": ["boolean", "null"], "description": "True if certificate states material is conforming/satisfactory"},
        "signatory":            {"type": ["string", "null"], "description": "Name of the authorised signatory"},
        "third_party_inspector":{"type": ["string", "null"], "description": "Third party inspector name and organisation — only present on 3.2 certs"}
    }
}

# Wrapped in the format the Mistral Document AI API expects
DOCUMENT_ANNOTATION_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "en10204_cert",
        "schema": EN10204_SCHEMA
    }
}

print("Schema fields:", list(EN10204_SCHEMA["properties"].keys()))

Schema fields: ['cert_number', 'cert_type', 'issue_date', 'issuer', 'customer', 'purchase_order', 'product', 'melt_practice', 'chemical_analysis', 'mechanical_properties', 'impact_test', 'hardness', 'heat_treatment', 'grain_size', 'conforming', 'signatory', 'third_party_inspector']


## Helper functions

In [3]:
def encode_pdf(path: Path) -> str:
    """PDF must be base64 encoded to send as JSON payload to the API."""
    return base64.b64encode(path.read_bytes()).decode()


def call_ocr(path: Path, pages: list[int], extract_fields: bool = False) -> dict:
    """
    Call Mistral Document AI for specific pages of a PDF.

    pages           - 0-indexed list of pages to process. Only send cert pages,
                      not enclosures (UT reports, NDE rosters, scan plans).
    extract_fields  - When True, adds document_annotation_format to the request.
                      The API then extracts structured EN 10204 fields alongside
                      the OCR text in the same call — no second model needed.
    """
    payload = {
        "model": DEPLOYMENT,
        "document": {
            "type": "document_url",
            "document_url": f"data:application/pdf;base64,{encode_pdf(path)}"
        },
        "pages": pages,
        "include_image_base64": False
    }
    if extract_fields:
        payload["document_annotation_format"] = DOCUMENT_ANNOTATION_FORMAT

    response = requests.post(url=OCR_ENDPOINT, json=payload, headers=HEADERS)
    response.raise_for_status()
    return response.json()


def get_full_text(ocr_result: dict) -> str:
    """Joins markdown from all pages into one string for pattern matching."""
    return "\n\n".join(page["markdown"] for page in ocr_result["pages"])


def save_ocr_markdown(pdf_path: Path, ocr_result: dict) -> Path:
    out = OCR_DIR / f"{pdf_path.stem}.md"
    text = "\n\n---\n\n".join(page["markdown"] for page in ocr_result["pages"])
    out.write_text(text, encoding="utf-8")
    return out


def save_annotation(pdf_path: Path, data: dict) -> Path:
    out = ANNOTATION_DIR / f"{pdf_path.stem}.json"
    out.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    return out

## Cert type and page count detection

Run on page 0 only (cheap — 1 page). Determines:
- Whether to proceed (3.1/3.2) or stop (2.1/2.2 have no test data)
- How many pages the primary cert spans so we don't OCR enclosures

In [ ]:
# Covers all EN 10204 type declaration formats seen in real certs:
# "EN 10204 3.1", "BSEN 10204:2004 3.1", "DIN EN 10204/01.05 3.1",
# "INSPECTION CERTIFICATE TYPE: EN 10204 3.1"
# Note: BSEN has no space — (?:BS)? handles both "BS EN" and "BSEN"
_CERT_TYPE_RE = re.compile(
    r'(?:BS\s*)?(?:DIN\s+)?EN\s*10204.*?([23]\.[12])',
    re.IGNORECASE
)

# Covers page count markers across English, Italian, Spanish:
# "Page: 1/2", "1/1", "Pag 1 of 2", "1 de 2", "Sheet 1/1"
_PAGE_COUNT_RE = re.compile(
    r'(?:page|pag|sheet|foglio)?[\s:]*1\s*(?:/|of|de)\s*(\d+)',
    re.IGNORECASE
)

# EN 10204 primary certs never exceed 3 pages.
# Higher numbers would mean we've matched an enclosure page marker, not the cert.
MAX_CERT_PAGES = 3


def get_cert_type(page0_text: str) -> str | None:
    match = _CERT_TYPE_RE.search(page0_text)
    return match.group(1) if match else None


def get_cert_page_count(page0_text: str) -> int:
    """
    Returns how many pages belong to the primary cert.
    Defaults to 2 when no page marker is found — this covers the
    distributor wrap pattern where a cover page (no page number)
    sits in front of the mill cert, which starts on page 1.
    """
    match = _PAGE_COUNT_RE.search(page0_text)
    if match:
        return min(int(match.group(1)), MAX_CERT_PAGES)
    return 2

## Main extraction pipeline

In [5]:
def process(pdf_path: Path) -> dict:
    """
    Full extraction pipeline for one certificate PDF.

    Step 1 — Page 0 OCR (no annotation, cheap)
        Detect cert type. Stop here for 2.1/2.2 — they have no test data.
        Detect how many pages the primary cert spans.

    Step 2 — Cert pages OCR + annotation (one combined call)
        OCR only the cert pages (not enclosures).
        document_annotation_format extracts EN 10204 fields in the same call.
        The model sees page images + OCR text — better accuracy than text-only extraction.

    Returns dict with keys:
        cert_type    — detected EN 10204 type or None
        pages_ocrd   — number of cert pages processed
        skipped      — reason string if processing was stopped early, else None
        fields       — extracted EN 10204 fields (from document_annotation)
        ocr          — raw Mistral OCR response
    """
    print(f"\n{pdf_path.name}")

    # Step 1: page 0 only to detect cert type and page count
    print("  OCR page 0...")
    page0 = call_ocr(pdf_path, pages=[0], extract_fields=False)
    page0_text = page0["pages"][0]["markdown"]

    cert_type  = get_cert_type(page0_text)
    page_count = get_cert_page_count(page0_text)
    print(f"  Cert type: {cert_type or 'not detected'}  |  Cert pages: {page_count}")

    # 2.1/2.2 are declaration certs — no chemical analysis or mechanical test results
    if cert_type in ("2.1", "2.2"):
        return {"cert_type": cert_type, "pages_ocrd": 1,
                "skipped": f"EN 10204 {cert_type} — declaration cert, no test data",
                "fields": None, "ocr": page0}

    if cert_type is None:
        return {"cert_type": None, "pages_ocrd": 1,
                "skipped": "EN 10204 type not found on page 0 — check document",
                "fields": None, "ocr": page0}

    # Step 2: OCR cert pages with field extraction
    cert_pages = list(range(page_count))
    print(f"  OCR pages {cert_pages} + extracting fields...")
    result = call_ocr(pdf_path, pages=cert_pages, extract_fields=True)

    # document_annotation is a JSON string in the API response
    raw_annotation = result.get("document_annotation", "{}")
    fields = json.loads(raw_annotation) if isinstance(raw_annotation, str) else raw_annotation

    # Flag any mandatory fields the model could not find
    mandatory = ["cert_number", "chemical_analysis", "mechanical_properties", "heat_treatment"]
    missing   = [f for f in mandatory if not fields.get(f)]
    if missing:
        print(f"  ⚠  Not found in cert: {missing}")
    else:
        print("  ✓  All mandatory fields extracted")

    return {
        "cert_type":  cert_type,
        "pages_ocrd": page_count,
        "skipped":    None,
        "fields":     fields,
        "ocr":        result
    }

## List available PDFs

In [6]:
pdf_files = sorted(INPUT_DIR.glob("*.pdf"))
for i, p in enumerate(pdf_files):
    print(f"[{i}] {p.name}  ({p.stat().st_size // 1024} KB)")

[0] 147149__02 MELESI MA-00351 mds-DH44.pdf  (5179 KB)
[1] EUR-472273.pdf  (4135 KB)
[2] FMT-22-1454-EZ17.pdf  (1499 KB)


## Process one file
Change `FILE_INDEX` to pick a different PDF from the list above.

In [7]:
FILE_INDEX = 0

target = pdf_files[FILE_INDEX]
result = process(target)

if result["skipped"]:
    print(f"\nSkipped: {result['skipped']}")
else:
    ocr_path  = save_ocr_markdown(target, result["ocr"])
    ann_path  = save_annotation(target, result["fields"])
    print(f"\nOCR saved    : {ocr_path.relative_to(OUTPUT_DIR)}")
    print(f"Fields saved : {ann_path.relative_to(OUTPUT_DIR)}")


147149__02 MELESI MA-00351 mds-DH44.pdf
  OCR page 0...


  Cert type: 3.1  |  Cert pages: 1
  OCR pages [0] + extracting fields...


  ✓  All mandatory fields extracted

OCR saved    : ocr-mistral-document-ai-2505-1\147149__02 MELESI MA-00351 mds-DH44.md
Fields saved : annotation-v1\147149__02 MELESI MA-00351 mds-DH44.json


## Preview extracted fields

In [8]:
if result["fields"]:
    display(Markdown("```json\n" + json.dumps(result["fields"], indent=2, ensure_ascii=False) + "\n```"))
else:
    print(result.get("skipped", "No fields extracted"))

```json
{
  "cert_number": "191621 Rev.01",
  "cert_type": "3.1",
  "issue_date": "04/04/24",
  "issuer": "P. OFF. A. MELESI & C.",
  "customer": "FRAMO FLATOY AS",
  "purchase_order": "1002-PO-013388",
  "product": {
    "form": "Premachined Ring",
    "grade": "F51/F60",
    "uns": null,
    "specification": [
      "ASTM A182",
      "QTR Nr. 1",
      "MDS D44",
      "MS05"
    ],
    "dimensions": [
      "Øe473xØi300xH145",
      "Øe473xØi174xH153",
      "Øe471xØi304xH78"
    ],
    "quantity": 6
  },
  "melt_practice": "A.O.D.",
  "chemical_analysis": [
    {
      "heat": "278287",
      "C": 0.014,
      "Si": 0.39,
      "Mn": 1.18,
      "P": 0.026,
      "S": 0.0,
      "Cr": 22.47,
      "Ni": 5.71,
      "Mo": 3.24,
      "N": 0.18,
      "Nb_Ta": null,
      "Co": null,
      "Cu": null,
      "Fe": null,
      "Al": null,
      "Ti": null
    },
    {
      "heat": "300309",
      "C": 0.021,
      "Si": 0.44,
      "Mn": 1.19,
      "P": 0.02,
      "S": 0.0,
      "Cr": 22.46,
      "Ni": 5.76,
      "Mo": 3.2,
      "N": 0.18,
      "Nb_Ta": null,
      "Co": null,
      "Cu": null,
      "Fe": null,
      "Al": null,
      "Ti": null
    },
    {
      "heat": "377897",
      "C": 0.024,
      "Si": 0.37,
      "Mn": 1.21,
      "P": 0.025,
      "S": 0.0,
      "Cr": 22.46,
      "Ni": 5.62,
      "Mo": 3.17,
      "N": 0.18,
      "Nb_Ta": null,
      "Co": null,
      "Cu": null,
      "Fe": null,
      "Al": null,
      "Ti": null
    }
  ],
  "mechanical_properties": [
    {
      "heat": "278287",
      "yield_mpa": 549,
      "uts_mpa": 739,
      "elongation_pct": 37.48,
      "reduction_area_pct": 72.47
    },
    {
      "heat": "300309",
      "yield_mpa": 516,
      "uts_mpa": 686,
      "elongation_pct": 35.44,
      "reduction_area_pct": 71.78
    },
    {
      "heat": "377897",
      "yield_mpa": 536,
      "uts_mpa": 715,
      "elongation_pct": 36.66,
      "reduction_area_pct": 71.91
    }
  ],
  "impact_test": [
    {
      "heat": "278287",
      "temp_c": -46,
      "energy_j": [
        159,
        164,
        171
      ]
    },
    {
      "heat": "300309",
      "temp_c": -46,
      "energy_j": [
        163,
        172,
        179
      ]
    },
    {
      "heat": "377897",
      "temp_c": -46,
      "energy_j": [
        162,
        169,
        175
      ]
    }
  ],
  "hardness": [
    {
      "heat": "278287",
      "method": "Brinell",
      "values": [
        159,
        164,
        171
      ]
    },
    {
      "heat": "300309",
      "method": "Brinell",
      "values": [
        163,
        172,
        179
      ]
    },
    {
      "heat": "377897",
      "method": "Brinell",
      "values": [
        162,
        169,
        175
      ]
    }
  ],
  "heat_treatment": {
    "type": "Solution Annealed",
    "temp_c": 1060,
    "soak": "min 1/2 h/inch",
    "cooling": "quenched water"
  },
  "grain_size": null,
  "conforming": true,
  "signatory": "Andrea Gubetti",
  "third_party_inspector": null
}
```

## Preview OCR text

In [9]:
if result["ocr"]:
    for i, page in enumerate(result["ocr"]["pages"]):
        display(Markdown(f"### Page {i + 1}"))
        display(Markdown(page["markdown"]))

### Page 1

| 23813 Cortenova (Lecco)
ITALY
Largo De Vecchi 11
P.o. box 11
Tel. 0341.98341
Fax 0341.983496 - 983497 | CERTIFICATO DI COLLAUDO
INSPECTION CERTIFICATE
CERTIFICAT D'ESSAIS | N. DATA / DATE SHEET /
191621 Rev.01 04/04/24 1/1 |
| --- | --- | --- |
| EN 10204 3.1 |
| ORDINE / ORDER / C.DE
1002-PO-013388 |  |
| DATA / DATE
16/01/24 | COMMESSA
75698 | FRAMO FLATOY AS
FLATØYVEGEN 24
NO-59 FREKHAUG NORWAY N |
| N. RIF. | ITEM | N / QUANTITY | DESCRIZIONE / DESCRIPTION / DESIGNATION | NOTE |
| SR1852 | 20 | 2 | F51/F60 PREMACHINED RING | mm Øe473xØi300xH145
QTR Nr. 1 / MDS D44 / MS05 |
| SR1870 | 40 | 2 | F51/F60 PREMACHINED RING | mm Øe473xØi174xH153
QTR Nr. 1 / MDS D44 / MS05 |
| SR1906 | 60 | 2 | F51/F60 PREMACHINED RING | mm Øe471xØi304xH78
QTR Nr. 1 / MDS D44 / MS05 |
| N. | MATERIALE / MATERIAL / MATÉRIEL | HEAT / COULÉE | ACCIAIERIA / STEEL PLANT / ACIÉRIE | ORIGIN. CERT. / CERT. D'ORIGIN |
| SR1852 | ASTM A182 F51/F60 | 278287 | COGNE Refining method A.O.D. | 2022049752 |
| SR1870 | ASTM A182 F51/F60 | 300309 | COGNE Refining method A.O.D. | 2023008195 |
| SR1906 | ASTM A182 F51/F60 | 377897 | COGNE Refining method A.O.D. | 2023053680 |
| N. RIF. | ANALISI CHIMICA / CHEMICAL ANALYSIS / COMPOSITION CHIMIQUE |
| C | Mn | Si | P | S | Cr | Ni | Mo | N |  |  |
| Min | 0,030 | 2,000 | 1,000 | 0,030 | 0,020 | 21,000 | 4,500 | 2,500 | 0,140 |  |  |
| Max |  |  |  |  |  | 23,000 | 6,500 | 3,500 | 0,200 |  |  |
| SR1852Lad | 0,014 | 1,180 | 0,390 | 0,026 | 0,000 | 22,470 | 5,710 | 3,240 | 0,180 |  |  |
| SR1852Chec | 0,014 | 1,200 | 0,394 | 0,026 | 0,000 | 22,490 | 5,730 | 3,250 | 0,180 |  |  |
| SR1870Lad | 0,021 | 1,190 | 0,440 | 0,020 | 0,000 | 22,460 | 5,760 | 3,200 | 0,180 |  |  |
| SR1870Chec | 0,021 | 1,190 | 0,441 | 0,020 | 0,000 | 22,460 | 5,760 | 3,200 | 0,180 |  |  |
| SR1906Lad | 0,024 | 1,210 | 0,370 | 0,025 | 0,000 | 22,460 | 5,620 | 3,170 | 0,180 |  |  |
| SR1906Chec | 0,023 | 1,120 | 0,352 | 0,024 | 0,000 | 22,370 | 5,530 | 3,120 | 0,177 |  |  |
| N. RIF. | PROVE MECCANICHE / MECHANICAL TESTS / ESSAIS MECANIQUES |
| ømm. | SO mm.2 | LU mm. | RS.-YIELD
LIMITE | R.-TENSILE
RUPTURE | A. %-ELONG.
ALLONGEM. | C. %-REDUCT.
STRICTION | RESILIENZA / IMPACT TEST / RESILIENCE | HARDNESS / DURETE |
| MEDIO / AVERAGE
MOYEN
Joule | MINIMO
MINIMUM
Joule | TIPO
TYPE | TEMP.
°C |
| Brinell
10-3000-30
Max.271 |
| A1126TRAN | 12,50 | 122,65 | 50,00 | 549,00 | 739,00 | 37,48 | 72,47 | 159 | 164 | 171 | KV - 46 |
| A1126LONG | 12,50 | 122,65 | 50,00 | 516,00 | 686,00 | 35,44 | 71,78 | 163 | 172 | 179 | KV - 46 |
| A1008TRAN | 12,50 | 122,65 | 50,00 | 545,00 | 705,00 | 38,17 | 65,60 | 157 | 164 | 173 | KV - 46 |
| A1008LONG | 12,50 | 122,65 | 50,00 | 488,00 | 723,00 | 36,49 | 67,53 | 166 | 175 | 182 | KV - 46 |
| A1261TRAN | 12,50 | 122,65 | 50,00 | 536,00 | 715,00 | 36,66 | 71,91 | 162 | 169 | 175 | KV - 46 |
| A1261LONG | 12,50 | 122,65 | 50,00 | 586,00 | 710,00 | 42,27 | 79,22 | 167 | 176 | 184 | KV - 46 |
| N. RIF. | TRATTAMENTI TERMICI / HEAT TREATMENTS / TRAITEMENTS THERMIQUES |
| SR1852 | Solution Annealed at 1060°C
for min 1/2 h/inch quenched water |
| SR1870 | Solution Annealed at 1060°C
for min 1/2 h/inch quenched water |
| SR1906 | Solution Annealed at 1060°C
for min 1/2 h/inch quenched water |
| EN 10204 3.1
Surface and dimensional check according to relevant standard.
Steel made by electric furnace.
Corrosion test acc. to ASTM G48 method "A" at 25°C / 24 hours, no weight loss, no pitting at 20x
Ferrite determined using ferritoscope fisher MP30 on all pieces, results: 35-60% on total body.
Material acc. to QTR 01 R8; Norsok M630 2020, MDS D44 Rev.6; MDS DH44 Rev.1 as applicable
No weld repair.
Reduction ratio > 4:1
water temperature before quenching < 40°C water temperature after quenching < 50°C
Mechanical tests according to ASTM A370
Test specimen orientation: transversal and longitudinal
Impact test specimen size 10 x 10 x 55 mm
The works operate a quality management system acc. to annex I, section 4.3 of the pressure
equipment directive (PED) 2014/68/EU certificate by TUV certificate nr 58/2002/MUC.
Material acc. to MA-00351 REV F |
| ESITO DELLE PROVE / TEST ISSUE / RESULTS DES ESSAIS: FAVOREVOLE SATISFACTORY SATISFAISANT |
| TRADE MARK | P. OFF. A. MELESI & C.
Electronically Generated Certificate | COLLAUDATORE / INSPECTOR / INSPECTEUR
Andrea Gubetti | ENTE COLLAUDATORE / INSPECTION AGENCY
AUTHORITÉE DE RECETTE |

## Batch — process all PDFs
Each file makes 1–2 API calls. Only run when you want to process everything.

In [10]:
for pdf in pdf_files:
    try:
        r = process(pdf)
        if r["skipped"]:
            print(f"  Skipped: {r['skipped']}")
        else:
            save_ocr_markdown(pdf, r["ocr"])
            save_annotation(pdf, r["fields"])
            print(f"  Saved to {OCR_DIR.name}/ and {ANNOTATION_DIR.name}/")
    except requests.HTTPError as e:
        print(f"  API error {e.response.status_code}: {e.response.text[:200]}")


147149__02 MELESI MA-00351 mds-DH44.pdf
  OCR page 0...


  Cert type: 3.1  |  Cert pages: 1
  OCR pages [0] + extracting fields...


  ✓  All mandatory fields extracted
  Saved to ocr-mistral-document-ai-2505-1/ and annotation-v1/

EUR-472273.pdf
  OCR page 0...


  Cert type: 3.1  |  Cert pages: 3
  OCR pages [0, 1, 2] + extracting fields...


  ✓  All mandatory fields extracted
  Saved to ocr-mistral-document-ai-2505-1/ and annotation-v1/

FMT-22-1454-EZ17.pdf
  OCR page 0...


  Cert type: 3.1  |  Cert pages: 2
  OCR pages [0, 1] + extracting fields...


  ✓  All mandatory fields extracted
  Saved to ocr-mistral-document-ai-2505-1/ and annotation-v1/
